# PDF to XML Conversion

Converts PDF files to structured TEI XML using [GROBID](https://github.com/kermitt2/grobid), then detects the language of each document and separates English-only files into a dedicated folder.

**Input:** `pdfs_final/` — merged PDF collection produced by `OCR.ipynb`  
**Output:**
- `output/xml_all/` — TEI XML files for all languages
- `output/xml_english/` — English-only subset (input for `ExtractXML.ipynb`)

**Prerequisites:** GROBID must be running before executing this notebook:
```
docker run -t --rm -p 8070:8070 lfoppiano/grobid:0.8.0
```

In [ ]:
import shutil
import threading
import time
import xml.etree.ElementTree as ET
from datetime import datetime
from pathlib import Path
from typing import List, Optional

import requests
from grobid_client.grobid_client import GrobidClient
from lingua import LanguageDetectorBuilder

In [ ]:
PROJECT_ROOT = Path("..").resolve()

PDFS_DIR    = PROJECT_ROOT / "pdfs_final"       # input PDFs
OUTPUT_DIR  = PROJECT_ROOT / "output"
XML_ALL_DIR = OUTPUT_DIR / "xml_all"            # all languages
XML_EN_DIR  = OUTPUT_DIR / "xml_english"        # English only

GROBID_SERVER = "http://localhost:8070"         # change if GROBID runs on a different host
GROBID_THREADS = 10                             # concurrent processing threads

In [ ]:
class PDFToXMLProcessor:
    """
    Converts a folder of PDFs to TEI XML via GROBID and separates English documents.
    Produces two output folders: xml_all (all languages) and xml_english (English only).
    """

    def __init__(self, input_dir: Path, output_dir: Path, grobid_server: str = GROBID_SERVER):
        self.input_dir  = input_dir
        self.output_dir = output_dir
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # Verify GROBID is reachable before doing any work
        try:
            response = requests.get(f"{grobid_server}/api/isalive", timeout=5)
            if response.status_code != 200:
                raise ConnectionError("Server returned non-200 status")
            print(f"GROBID server OK: {grobid_server}")
        except Exception as e:
            print(f"GROBID server not available at {grobid_server}: {e}")
            print("Start it with: docker run -t --rm -p 8070:8070 lfoppiano/grobid:0.8.0")
            raise

        self.client = GrobidClient(
            grobid_server=grobid_server,
            batch_size=1000,
            coordinates=["persName", "biblStruct", "title", "head"],
            sleep_time=5,
            timeout=60,
            check_server=True,
        )
        self.language_detector = LanguageDetectorBuilder.from_all_languages().build()
        self.stats = {"total": 0, "processed": 0, "english": 0, "non_english": 0, "errors": 0, "languages": {}}

        print(f"Input:  {self.input_dir}")
        print(f"Output: {self.output_dir} (xml_all / xml_english)")

    def detect_language_from_xml(self, xml_path: Path) -> Optional[str]:
        """
        Detect document language from a TEI XML file using lingua-py.
        Samples title, abstract, and the first 5 body paragraphs.

        Returns an ISO 639-1 code (e.g. 'en', 'de') or None if detection fails.
        """
        try:
            root = ET.parse(xml_path).getroot()
            ns = {"tei": "http://www.tei-c.org/ns/1.0"}
            parts = []

            title = root.find(".//tei:titleStmt/tei:title", ns)
            if title is not None and title.text:
                parts.append(title.text.strip())

            abstract = root.find(".//tei:abstract", ns)
            if abstract is not None:
                text = "".join(abstract.itertext()).strip()
                if text:
                    parts.append(text)

            body = root.find(".//tei:body", ns)
            if body is not None:
                for p in body.findall(".//tei:p", ns)[:5]:
                    text = "".join(p.itertext()).strip()
                    if text:
                        parts.append(text)

            combined = " ".join(parts)[:1000]
            if len(combined) > 20:
                lang = self.language_detector.detect_language_of(combined)
                if lang:
                    return lang.iso_code_639_1.name.lower()
        except Exception:
            pass
        return None

    def process_pdfs(self, verbose: bool = False, clean_before: bool = True) -> None:
        """
        Run GROBID on all PDFs in input_dir, then filter English documents.

        Args:
            verbose:      Print per-file errors during language detection.
            clean_before: Delete previous xml_all / xml_english output before starting.
        """
        xml_all_dir = self.output_dir / "xml_all"
        xml_en_dir  = self.output_dir / "xml_english"

        if clean_before:
            for d in [xml_all_dir, xml_en_dir]:
                if d.exists():
                    print(f"Clearing {d}...")
                    shutil.rmtree(d)

        xml_all_dir.mkdir(exist_ok=True)
        xml_en_dir.mkdir(exist_ok=True)

        pdf_files = list(self.input_dir.glob("*.pdf"))
        self.stats["total"] = len(pdf_files)

        if not pdf_files:
            print(f"No PDF files found in {self.input_dir}")
            return

        print(f"\nFound {self.stats['total']} PDF files — starting GROBID...")
        start_time = time.time()

        # Run GROBID in a background thread so we can monitor progress
        def run_grobid():
            self.client.process(
                "processFulltextDocument",
                str(self.input_dir),
                output=str(xml_all_dir),
                n=GROBID_THREADS,
                force=True,
            )

        grobid_thread = threading.Thread(target=run_grobid, daemon=True)
        grobid_thread.start()

        last_count = 0
        while grobid_thread.is_alive():
            time.sleep(10)
            current_count = len(list(xml_all_dir.glob("*.tei.xml")))
            if current_count > last_count:
                elapsed  = time.time() - start_time
                speed    = current_count / elapsed if elapsed > 0 else 0
                remaining = (self.stats["total"] - current_count) / speed if speed > 0 else 0
                print(
                    f"[{datetime.now().strftime('%H:%M:%S')}] "
                    f"{current_count}/{self.stats['total']} "
                    f"({current_count * 100 // self.stats['total']}%) | "
                    f"{speed:.1f} files/sec | "
                    f"elapsed {time.strftime('%H:%M:%S', time.gmtime(elapsed))} | "
                    f"remaining ~{time.strftime('%H:%M:%S', time.gmtime(remaining))}"
                )
                last_count = current_count

        grobid_thread.join()
        total_time = time.strftime("%H:%M:%S", time.gmtime(time.time() - start_time))
        print(f"\nGROBID done in {total_time}")

        # Language detection
        xml_files = list(xml_all_dir.glob("*.tei.xml"))
        if not xml_files:
            print("GROBID produced no XML files — check server logs")
            return

        print(f"\nDetecting language for {len(xml_files)} files...")
        for i, xml_file in enumerate(xml_files, 1):
            self.stats["processed"] += 1
            try:
                lang = self.detect_language_from_xml(xml_file)
                if lang:
                    self.stats["languages"][lang] = self.stats["languages"].get(lang, 0) + 1
                if lang == "en":
                    shutil.copy2(xml_file, xml_en_dir / xml_file.name)
                    self.stats["english"] += 1
                else:
                    self.stats["non_english"] += 1
            except Exception as e:
                self.stats["errors"] += 1
                if verbose:
                    print(f"  Error in {xml_file.name}: {e}")

            if i % 100 == 0 or i == len(xml_files):
                print(f"  {i}/{len(xml_files)} ({i * 100 // len(xml_files)}%) | English so far: {self.stats['english']}")

    def print_statistics(self) -> None:
        """Print a summary of processing results."""
        print("\n" + "=" * 60)
        print(f"Total PDFs:       {self.stats['total']}")
        print(f"Processed (XML):  {self.stats['processed']}")
        print(f"English:          {self.stats['english']}")
        print(f"Other languages:  {self.stats['non_english']}")
        print(f"Errors:           {self.stats['errors']}")

        if self.stats["processed"] > 0:
            print(f"\nSuccess rate: {self.stats['processed'] / self.stats['total'] * 100:.1f}%")
            print(f"English rate: {self.stats['english'] / self.stats['processed'] * 100:.1f}%")

        if self.stats["languages"]:
            print("\nTop languages:")
            for lang, count in sorted(self.stats["languages"].items(), key=lambda x: -x[1])[:10]:
                pct = count / self.stats["processed"] * 100
                print(f"  {lang}: {count} ({pct:.1f}%)")

        print(f"\nOutput: {self.output_dir / 'xml_all'} ({self.stats['processed']} files)")
        print(f"        {self.output_dir / 'xml_english'} ({self.stats['english']} files)")
        print("=" * 60)

    def get_english_files(self) -> List[Path]:
        """Return a list of English TEI XML file paths."""
        return list((self.output_dir / "xml_english").glob("*.tei.xml"))

    def get_all_files(self) -> List[Path]:
        """Return a list of all TEI XML file paths."""
        return list((self.output_dir / "xml_all").glob("*.tei.xml"))

In [ ]:
processor = PDFToXMLProcessor(input_dir=PDFS_DIR, output_dir=OUTPUT_DIR, grobid_server=GROBID_SERVER)
processor.process_pdfs(verbose=False, clean_before=True)
processor.print_statistics()

english_files = processor.get_english_files()
print(f"\nEnglish XML files ready: {len(english_files)}")